# Lab | Error Handling

## Exercise: Error Handling for Managing Customer Orders

The implementation of your code for managing customer orders assumes that the user will always enter a valid input. 

For example, we could modify the `initialize_inventory` function to include error handling.
   - If the user enters an invalid quantity (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the quantity for that product.
   - Use a try-except block to handle the error and continue prompting the user until a valid quantity is entered.

```python
# Step 1: Define the function for initializing the inventory with error handling
def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_quantity = False
        while not valid_quantity:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity < 0:
                    raise ValueError("Invalid quantity! Please enter a non-negative value.")
                valid_quantity = True
            except ValueError as error:
                print(f"Error: {error}")
        inventory[product] = quantity
    return inventory

# Or, in another way:

def initialize_inventory(products):
    inventory = {}
    for product in products:
        valid_input = False
        while not valid_input:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: "))
                if quantity >= 0:
                    inventory[product] = quantity
                    valid_input = True
                else:
                    print("Quantity cannot be negative. Please enter a valid quantity.")
            except ValueError:
                print("Invalid input. Please enter a valid quantity.")
    return inventory
```

Let's enhance your code by implementing error handling to handle invalid inputs.

Follow the steps below to complete the exercise:

2. Modify the `calculate_total_price` function to include error handling.
   - If the user enters an invalid price (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the price for that product.
   - Use a try-except block to handle the error and continue prompting the user until a valid price is entered.

3. Modify the `get_customer_orders` function to include error handling.
   - If the user enters an invalid number of orders (e.g., a negative value or a non-numeric value), display an error message and ask them to re-enter the number of orders.
   - If the user enters an invalid product name (e.g., a product name that is not in the inventory), or that doesn't have stock available, display an error message and ask them to re-enter the product name. *Hint: you will need to pass inventory as a parameter*
   - Use a try-except block to handle the error and continue prompting the user until a valid product name is entered.

4. Test your code by running the program and deliberately entering invalid quantities and product names. Make sure the error handling mechanism works as expected.


In [ ]:
from functools import reduce

def initialize_inventory(products):
    """1. Initializes the inventory dictionary with robust numeric validation loops."""
    inventory = {}
    print("\n--- Initialize Inventory ---")
    for product in products:
        valid_input = False
        while not valid_input:
            try:
                quantity = int(input(f"Enter the quantity of {product}s available: ").strip())
                if quantity >= 0:
                    inventory[product] = quantity
                    valid_input = True
                else:
                    print("Error: Quantity cannot be negative. Please try again.")
            except ValueError:
                print("Error: Invalid input! Please enter a valid whole number.")
    return inventory


def get_customer_orders(inventory):
    """3. Gathers orders with error handling for negative counts, missing items, and out-of-stock items."""
    print("\n--- Get Customer Orders ---")
    
    # Part A: Validate the total number of orders requested
    while True:
        try:
            num_orders = int(input("Enter the number of customer orders: ").strip())
            if num_orders < 0:
                print("Error: Number of orders cannot be negative. Please try again.")
                continue
            break
        except ValueError:
            print("Error: Invalid input! Please enter a valid whole number.")

    # Create a quick lowercase mapping to handle case-insensitive entries safely
    inventory_lowercase = {product.lower(): product for product in inventory}
    customer_orders = set()
    
    # Part B: Gather and validate individual product names
    for i in range(num_orders):
        while True:
            order = input(f"Enter the name of product #{i+1} that a customer wants to order: ").strip()
            order_lower = order.lower()
            
            # Check 1: Does the product exist in the store catalog?
            if order_lower not in inventory_lowercase:
                print(f"Error: '{order}' does not exist in our inventory catalog. Please try a different product.")
                continue
                
            original_name = inventory_lowercase[order_lower]
            
            # Check 2: Does the product have stock available right now?
            if inventory[original_name] <= 0:
                print(f"Error: '{original_name}' is currently out of stock! Please choose a different product.")
                continue
                
            # If all checks pass, add to order records and break the retry loop
            customer_orders.add(original_name)
            break
            
    return customer_orders


def update_inventory(customer_orders, inventory):
    """Updates inventory counts and uses a comprehension to filter out products with 0 stock."""
    # Deduct stock for chosen items (validation in previous step guarantees stock exists)
    for order in customer_orders:
        if order in inventory:
            inventory[order] -= 1

    # Dictionary Comprehension: Automatically filters out items whose quantity drops to 0
    updated_inventory = {prod: qty for prod, qty in inventory.items() if qty > 0}
    
    inventory.clear()
    inventory.update(updated_inventory)


def calculate_order_statistics(customer_orders, products):
    """Calculates order statistics using a list filter operation."""
    total_products_ordered = len(customer_orders)
    products_lower = [p.lower() for p in products]
    
    valid_orders_count = sum(1 for order in customer_orders if order.lower() in products_lower)
    
    percentage_ordered = (valid_orders_count / len(products)) * 100 if products else 0.0
    return total_products_ordered, percentage_ordered


def print_order_statistics(order_statistics):
    """Prints the calculated order statistics."""
    total, percentage = order_statistics
    print("\nOrder Statistics:")
    print(f"Total Products Ordered: {total}")
    print(f"Percentage of Unique Products Ordered: {percentage:.1f}%")


def print_updated_inventory(inventory):
    """Prints the updated inventory contents cleanly."""
    print("\nUpdated Inventory:")
    for product, quantity in inventory.items():
        print(f"{product}: {quantity}")


def calculate_total_price(customer_orders):
    """2 & 5. Prompts for product prices with error handling and prints the total sum."""
    print()  # Formatted padding spacing
    total_price = 0.0
    
    for product in customer_orders:
        while True:
            try:
                price = float(input(f"Enter the price of {product}: ").strip())
                if price < 0:
                    print("Error: Price cannot be negative. Please try again.")
                    continue
                total_price += price
                break
            except ValueError:
                print("Error: Invalid input! Please enter a valid decimal number (e.g., 10.50).")
                
    print(f"Total Price: ${total_price:.2f}")
    return total_price


# 7. Main program control flow
if __name__ == "__main__":
    # Base product master catalog array
    products = ["t-shirt", "mug", "hat", "book", "keychain"]
    
    # 1. Initialize
    inventory = initialize_inventory(products)
    
    # 2 & 3. Fetch orders (passing active inventory dictionary for cross-checking)
    customer_orders = get_customer_orders(inventory)
    
    # 4. Process analytics and modify system inventory counts
    order_statistics = calculate_order_statistics(customer_orders, products)
    update_inventory(customer_orders, inventory)
    
    # 5 & 6. Print final outputs out to console terminal
    print_order_statistics(order_statistics)
    print_updated_inventory(inventory)
    calculate_total_price(customer_orders)



--- Initialize Inventory ---
Error: Invalid input! Please enter a valid whole number.
Error: Invalid input! Please enter a valid whole number.
Error: Invalid input! Please enter a valid whole number.
Error: Quantity cannot be negative. Please try again.
Error: Quantity cannot be negative. Please try again.

--- Get Customer Orders ---
Error: 'pen' does not exist in our inventory catalog. Please try a different product.
Error: 'pencil' does not exist in our inventory catalog. Please try a different product.
Error: 'rubber' does not exist in our inventory catalog. Please try a different product.
Error: 'notebook' does not exist in our inventory catalog. Please try a different product.
Error: 'hoodie' does not exist in our inventory catalog. Please try a different product.
Error: 'letter' does not exist in our inventory catalog. Please try a different product.
Error: 'laptop' does not exist in our inventory catalog. Please try a different product.
Error: 'computer' does not exist in our 